# 2_train_gpu1
training our tuning fork networks, setting to use gpu0 to avoid memory conflict with other notebooks needing gpu allocation

In [1]:
#### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

#### custom
from InversePCA import InversePCA
from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

2024-02-21 17:13:02.843989: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-21 17:13:02.844020: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-21 17:13:02.844883: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-21 17:13:02.849656: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-02-21 17:13:03.432573: W tensorflow/compiler/tf2

Current GPU usage:
 - GPU0: 0B



2024-02-21 17:13:03.955224: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18447 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


## data prep
load in data and perform final prep (normalisation, label definition) before we start training)

In [2]:
#### load in grid
#df_full = pd.read_hdf('/home/oxs235/datastorage/repos_data/' + str(path) + '/data/df_all_log.h5', key='df')
df_full = pd.read_hdf('/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/data/keystone/barbie.h5', key='df')

#### define inputs
inputs = ['log_initial_mass_std', 'log_initial_Zinit_std', 'log_initial_Yinit_std', 'log_initial_MLT_std', 'log_star_age_std']

#### define outputs
classical_outputs = ['log_radius_std', 'log_luminosity_std', 'log_surface_Z_std']
astero_outputs = ['log_nu_max_std', 'log_delta_nu_std']

outputs = classical_outputs+astero_outputs

#### train/test split
seed = 42

df_train = df_full.sample(frac=0.9, random_state=seed)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.1, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))

#### can't have too many describes
df_full.describe()

Training set:  641680
Validation set:  71298


,initial_mass,initial_Zinit,initial_Yinit,initial_MLT,star_age,radius,luminosity,effective_T,surface_Z,nu_max,...,log_initial_Zinit_std,log_initial_Yinit_std,log_initial_MLT_std,log_star_age_std,log_radius_std,log_luminosity_std,log_effective_T_std,log_surface_Z_std,log_nu_max_std,log_delta_nu_std
count,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,...,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05
mean,1.032772,0.017551,0.278219,1.800823,4.819359,1.151215,1.617280,5882.540532,0.015335,2668.940264,...,2.029420e-15,2.871597e-15,-3.906865e-15,-8.005959e-17,-3.372441e-17,-2.367884e-17,-1.117476e-14,-1.805476e-15,-1.563090e-15,3.756325e-15
std,0.103474,0.010244,0.022917,0.115516,3.459256,0.255999,1.031820,472.426889,0.009956,925.630210,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,0.800000,0.004004,0.240020,1.600098,0.018656,0.687935,0.271118,4701.918937,0.000323,763.276107,...,-1.978866e+00,-1.750248e+00,-1.804411e+00,-4.430341e+00,-2.331268e+00,-2.590249e+00,-2.767900e+00,-3.581863e+00,-3.071945e+00,-3.098951e+00
25%,0.952000,0.008522,0.258311,1.701172,2.003543,0.962485,0.850404,5532.257625,0.006906,1963.104691,...,-8.152904e-01,-8.593967e-01,-8.525382e-01,-3.959378e-01,-7.398188e-01,-7.382142e-01,-7.294871e-01,-4.760246e-01,-6.201559e-01,-6.153846e-01
50%,1.042000,0.015451,0.277295,1.801318,4.165208,1.101051,1.356173,5848.726977,0.013637,2649.618353,...,1.011573e-01,8.491942e-04,3.637859e-02,2.354230e-01,-1.024183e-01,1.790694e-02,-3.222205e-02,2.139363e-01,1.581779e-01,1.624791e-01
75%,1.119000,0.025461,0.297773,1.900928,7.085151,1.284915,2.107733,6207.138200,0.022965,3343.707797,...,8.703738e-01,8.651243e-01,8.727972e-01,6.937153e-01,6.294013e-01,7.322872e-01,7.132740e-01,7.425388e-01,7.620387e-01,7.535334e-01
max,1.200000,0.039924,0.319990,1.999951,13.999716,2.073636,9.833521,7732.911306,0.039918,5388.566188,...,1.563217e+00,1.737967e+00,1.661935e+00,1.281240e+00,2.897512e+00,3.227549e+00,3.468157e+00,1.303204e+00,2.000567e+00,2.124895e+00


## gridsearch parameters
define gridsearch parameters for the tuning/pitchfork setup, focus on hparams that alter overarching architecture

In [3]:
"""
DEFINE TARGET ARCHITECTURES FOR GRID SEARCH
Rerun after training to avoid "___ not iterable" errors
"""
stem_d_layers = [4]
stem_d_units = [64]

ctine_d_layers = [4]
ctine_d_units = [64]

atine_d_layers = [4]
atine_d_units = [64]

archs = pd.DataFrame(product(stem_d_layers, stem_d_units, ctine_d_layers, ctine_d_units, atine_d_layers, atine_d_units))

archs.columns = ['stem_d_layers', 'stem_d_units', 'ctine_d_layers', 'ctine_d_units', 'atine_d_layers', 'atine_d_units']

In [4]:
"""
        ________
_______/
       \________
| stem | tines |

"""


tf.keras.backend.clear_session()


for arch_i in range(len(archs)):
    tf.keras.backend.clear_session()
    arch = archs.iloc[[arch_i]]
    
    ######## stem
    #### input
    stem_input = keras.Input(shape=(len(inputs),))

    #### dense layers
    stem_d_layers = arch["stem_d_layers"].iloc[0]
    stem_d_units = arch["stem_d_units"].iloc[0]

    for stem_d_layer in range(stem_d_layers):
        if stem_d_layer == 0:
            stem = layers.Dense(stem_d_units, activation='relu')(stem_input)
            stem = layers.LayerNormalization()(stem)
        else:
            stem = layers.Dense(stem_d_units, activation='relu')(stem)
            stem = layers.LayerNormalization()(stem)

    ######## classical tine
    #### dense layers
    ctine_d_layers = arch["ctine_d_layers"].iloc[0]
    ctine_d_units = arch["ctine_d_units"].iloc[0]

    for ctine_d_layer in range(ctine_d_layers):
        if ctine_d_layer == 0:
            ctine = layers.Dense(ctine_d_units, activation='relu')(stem)
            ctine = layers.LayerNormalization()(ctine)
        else:
            ctine = layers.Dense(ctine_d_units, activation='relu')(ctine)
            ctine = layers.LayerNormalization()(ctine)

    #### output
    ctine_out = layers.Dense(len(classical_outputs),name='classical_outs')(ctine)


    ######## astero tine
    #### dense layers
    atine_d_layers = arch["atine_d_layers"].iloc[0]
    atine_d_units = arch["atine_d_units"].iloc[0]
    
    for atine_d_layer in range(atine_d_layers):
        if atine_d_layer == 0:
            atine = layers.Dense(atine_d_units, activation='relu')(stem)
            atine = layers.LayerNormalization()(atine)
        else:
            atine = layers.Dense(atine_d_units, activation='relu')(atine)
            atine = layers.LayerNormalization()(atine)

    #### output
    atine_out = layers.Dense(int(len(astero_outputs)))(atine)

    ######## construct and fit
    model = keras.Model(inputs=stem_input, outputs=[ctine_out, atine_out], name='tuning_fork')

    #### compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

        
    model.compile(loss='MSE', optimizer=optimizer)
    
    #### fit model
    arch_name = 'barbie1'

    log_dir = "/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/logs/keystone/" + arch_name

    def scheduler(epoch, lr):
        if lr < 5e-5:
            return lr
        else:
            return lr * tf.math.exp(-0.001)

    lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, verbose=0)
                                                       
    cp_callback = tf.keras.callbacks.ModelCheckpoint("/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/models/keystone/" + arch_name + ".h5",
                                                     monitor= 'val_loss',
                                                     save_best_only= True,
                                                     save_freq='epoch')    

    tb_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir) 

    history = model.fit(df_train_inputs,
                        [df_train_outputs[classical_outputs],df_train_outputs[astero_outputs]],
                        validation_data=(df_val_inputs,[df_val_outputs[classical_outputs], df_val_outputs[astero_outputs]]),
                        batch_size=2**12,
                        verbose=1,
                        epochs=10000,
                        callbacks=[lr_callback, cp_callback, tb_callback],
                        shuffle=True
                       )  

2024-02-21 17:13:13.176070: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Epoch 1/10000


2024-02-21 17:13:15.366939: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-02-21 17:13:15.619460: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fa85c03dad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-02-21 17:13:15.619495: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2024-02-21 17:13:15.624695: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1708535595.701365  127946 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


157/157 [==============================] - 6s 10ms/step - loss: 0.3971 - classical_outs_loss: 0.1933 - dense_12_loss: 0.2038 - val_loss: 0.1060 - val_classical_outs_loss: 0.0539 - val_dense_12_loss: 0.0521 - lr: 9.9900e-04
Epoch 2/10000
 14/157 [=>............................] - ETA: 1s - loss: 0.0990 - classical_outs_loss: 0.0502 - dense_12_loss: 0.0488

/home/oxs235/miniconda3/envs/pitchfork/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


157/157 [==============================] - 2s 12ms/step - loss: 0.0580 - classical_outs_loss: 0.0309 - dense_12_loss: 0.0272 - val_loss: 0.0333 - val_classical_outs_loss: 0.0187 - val_dense_12_loss: 0.0146 - lr: 9.9800e-04
Epoch 3/10000
157/157 [==============================] - 2s 12ms/step - loss: 0.0256 - classical_outs_loss: 0.0144 - dense_12_loss: 0.0113 - val_loss: 0.0197 - val_classical_outs_loss: 0.0112 - val_dense_12_loss: 0.0085 - lr: 9.9700e-04
Epoch 4/10000
157/157 [==============================] - 2s 11ms/step - loss: 0.0168 - classical_outs_loss: 0.0095 - dense_12_loss: 0.0074 - val_loss: 0.0145 - val_classical_outs_loss: 0.0081 - val_dense_12_loss: 0.0063 - lr: 9.9601e-04
Epoch 5/10000
157/157 [==============================] - 2s 11ms/step - loss: 0.0128 - classical_outs_loss: 0.0072 - dense_12_loss: 0.0057 - val_loss: 0.0125 - val_classical_outs_loss: 0.0064 - val_dense_12_loss: 0.0061 - lr: 9.9501e-04
Epoch 6/10000
157/157 [==============================] - 2s 11ms/s

KeyboardInterrupt: 